# Dataset Inspection Notebook

In-depth inspection for a generated `geograph-lcm` dataset.

This notebook goes beyond `notebooks/eda_labels.py` and focuses on:
- schema and missingness
- duplicate keys and integrity checks
- temporal and spatial distributions
- class/year interaction
- image file health and resolution analysis
- random and per-class visual spot checks

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

%matplotlib inline

In [ ]:
# --- Config ---
DATASET_ROOT = Path("outputs/dataset")
LABELS_PATH = DATASET_ROOT / "labels.csv"
IMAGES_ROOT = DATASET_ROOT / "images"
RANDOM_SEED = 42
MAX_IMAGE_SCAN = None  # set e.g. 5000 for faster scans on very large datasets

assert LABELS_PATH.exists(), f"labels.csv not found: {LABELS_PATH}"
df = pd.read_csv(LABELS_PATH)

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
df.head(3)

In [ ]:
# --- Schema / missingness ---
required_cols = [
    "id", "image_path", "lat", "lon", "lcm_l2", "lcm_l3",
    "timestamp", "license", "sha256", "georef_confidence",
]

missing_required_cols = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing_required_cols if missing_required_cols else "None")

null_counts = df.isna().sum().sort_values(ascending=False)
empty_counts = (df.astype(str).apply(lambda s: s.str.strip().eq("")).sum()).sort_values(ascending=False)

quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": null_counts,
    "empty_string_count": empty_counts,
})
quality["null_pct"] = (quality["null_count"] / len(df) * 100.0).round(2)
quality["empty_pct"] = (quality["empty_string_count"] / len(df) * 100.0).round(2)
quality.sort_values(["null_count", "empty_string_count"], ascending=False)

In [ ]:
# --- Duplicate checks ---
id_dupes = df[df.duplicated(subset=["id"], keep=False)].sort_values("id") if "id" in df.columns else pd.DataFrame()
sha_dupes = df[df.duplicated(subset=["sha256"], keep=False)].sort_values("sha256") if "sha256" in df.columns else pd.DataFrame()

print(f"Duplicate id rows: {len(id_dupes):,}")
print(f"Duplicate sha256 rows: {len(sha_dupes):,}")

display(id_dupes[["id", "image_path", "lcm_l3"]].head(20) if not id_dupes.empty else pd.DataFrame({"note":["No duplicate ids"]}))
display(sha_dupes[["sha256", "id", "image_path"]].head(20) if not sha_dupes.empty else pd.DataFrame({"note":["No duplicate sha256"]}))

In [ ]:
# --- Label / confidence / license distributions ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

l3_counts = df["lcm_l3"].fillna("").astype(str).str.strip()
l3_counts = l3_counts[l3_counts != ""].value_counts().head(25)
axes[0,0].bar(l3_counts.index, l3_counts.values)
axes[0,0].set_title("Top LCM L3 classes")
axes[0,0].tick_params(axis="x", rotation=70)

l2_counts = df["lcm_l2"].fillna("").astype(str).str.strip()
l2_counts = l2_counts[l2_counts != ""].value_counts()
axes[0,1].bar(l2_counts.index, l2_counts.values)
axes[0,1].set_title("LCM L2 classes")
axes[0,1].tick_params(axis="x", rotation=45)

conf_counts = df["georef_confidence"].fillna("").astype(str).str.strip()
conf_counts = conf_counts[conf_counts != ""].value_counts()
axes[1,0].bar(conf_counts.index, conf_counts.values)
axes[1,0].set_title("Georeference confidence")

lic_counts = df["license"].fillna("").astype(str).str.strip()
lic_counts = lic_counts[lic_counts != ""].value_counts().head(12)
axes[1,1].bar(range(len(lic_counts)), lic_counts.values)
axes[1,1].set_title("Top license values")
axes[1,1].set_xticks(range(len(lic_counts)))
axes[1,1].set_xticklabels(lic_counts.index, rotation=60, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# --- Temporal diagnostics ---
ts = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
year_counts = ts.dt.year.value_counts().sort_index()
month_counts = ts.dt.to_period("M").value_counts().sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].bar(year_counts.index.astype(str), year_counts.values)
axes[0].set_title("Capture year distribution")

axes[1].plot(month_counts.index.astype(str), month_counts.values)
axes[1].set_title("Capture month trend")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

print("Timestamp parse success rate:", f"{(ts.notna().mean()*100):.2f}%")

In [ ]:
# --- Spatial diagnostics ---
lat = pd.to_numeric(df["lat"], errors="coerce")
lon = pd.to_numeric(df["lon"], errors="coerce")
mask = lat.notna() & lon.notna()
lat_v = lat[mask]
lon_v = lon[mask]

print(f"Rows with valid coordinates: {mask.sum():,} / {len(df):,}")
print("Latitude range:", (lat_v.min(), lat_v.max()))
print("Longitude range:", (lon_v.min(), lon_v.max()))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(lon_v, lat_v, s=6, alpha=0.4)
axes[0].set_title("Coordinate scatter")
axes[0].set_xlabel("lon")
axes[0].set_ylabel("lat")

hb = axes[1].hexbin(lon_v, lat_v, gridsize=60, cmap="viridis", mincnt=1)
axes[1].set_title("Coordinate density (hexbin)")
axes[1].set_xlabel("lon")
axes[1].set_ylabel("lat")
fig.colorbar(hb, ax=axes[1], label="count")

plt.tight_layout()
plt.show()

In [ ]:
# --- Image file health + resolution stats ---
scan_df = df.copy()
if MAX_IMAGE_SCAN is not None:
    scan_df = scan_df.head(MAX_IMAGE_SCAN)

records = []
for _, row in scan_df.iterrows():
    p = Path(str(row.get("image_path", ""))).expanduser()
    rec = {"id": row.get("id"), "image_path": str(p), "exists": p.exists(), "width": np.nan, "height": np.nan, "mode": None}
    if p.exists():
        try:
            with Image.open(p) as im:
                rec["width"], rec["height"] = im.size
                rec["mode"] = im.mode
        except Exception:
            pass
    records.append(rec)

img_stats = pd.DataFrame(records)
img_stats["pixels"] = img_stats["width"] * img_stats["height"]
img_stats["aspect_ratio"] = img_stats["width"] / img_stats["height"]

print(f"Scanned images: {len(img_stats):,}")
print(f"Missing files: {(~img_stats['exists']).sum():,}")
display(img_stats.describe(include="all").T)

In [ ]:
# --- Resolution / ratio plots ---
valid_img = img_stats[img_stats["exists"] & img_stats["width"].notna() & img_stats["height"].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(valid_img["width"], bins=40)
axes[0].set_title("Image width")

axes[1].hist(valid_img["height"], bins=40)
axes[1].set_title("Image height")

axes[2].hist(valid_img["aspect_ratio"].replace([np.inf, -np.inf], np.nan).dropna(), bins=40)
axes[2].set_title("Aspect ratio")

plt.tight_layout()
plt.show()

low_res = valid_img[(valid_img["width"] < 640) | (valid_img["height"] < 480)]
print(f"Images below 640x480: {len(low_res):,}")
display(low_res.head(20))

In [ ]:
# --- Class x Year heatmap (top classes) ---
temp = df.copy()
temp["year"] = pd.to_datetime(temp["timestamp"], errors="coerce", utc=True).dt.year
temp["lcm_l3"] = temp["lcm_l3"].fillna("").astype(str).str.strip()
temp = temp[temp["lcm_l3"] != ""]

top_classes = temp["lcm_l3"].value_counts().head(12).index
cross = temp[temp["lcm_l3"].isin(top_classes)].pivot_table(index="lcm_l3", columns="year", values="id", aggfunc="count", fill_value=0)

plt.figure(figsize=(12, 6))
plt.imshow(cross.values, aspect="auto", interpolation="nearest", cmap="magma")
plt.colorbar(label="count")
plt.yticks(range(len(cross.index)), cross.index)
plt.xticks(range(len(cross.columns)), [str(c) for c in cross.columns], rotation=45)
plt.title("Top L3 classes by capture year")
plt.tight_layout()
plt.show()

cross

In [ ]:
# --- Random visual sample grid ---
rng = random.Random(RANDOM_SEED)
candidates = df[df["image_path"].astype(str).map(lambda p: Path(p).exists())].copy()
sample_n = min(20, len(candidates))
sample = candidates.sample(sample_n, random_state=RANDOM_SEED) if sample_n > 0 else candidates

cols = 5
rows = max(1, math.ceil(max(1, len(sample)) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
axes = np.array(axes).reshape(rows, cols)

for i in range(rows * cols):
    ax = axes.flat[i]
    ax.axis("off")
    if i >= len(sample):
        continue
    row = sample.iloc[i]
    p = Path(str(row["image_path"]))
    try:
        with Image.open(p) as im:
            ax.imshow(im)
        ax.set_title(f"{row['id']}\n{row.get('lcm_l3', '')}", fontsize=8)
    except Exception:
        ax.text(0.5, 0.5, "Unreadable", ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# --- Per-class visual spot checks ---
top_l3 = df["lcm_l3"].fillna("").astype(str).str.strip()
top_l3 = top_l3[top_l3 != ""].value_counts().head(6).index.tolist()

examples_per_class = 4
fig, axes = plt.subplots(len(top_l3), examples_per_class, figsize=(3*examples_per_class, 3*max(1, len(top_l3))))
if len(top_l3) == 1:
    axes = np.array([axes])

for r, cls in enumerate(top_l3):
    cls_rows = df[df["lcm_l3"].astype(str).str.strip() == cls]
    cls_rows = cls_rows[cls_rows["image_path"].astype(str).map(lambda p: Path(p).exists())].head(examples_per_class)
    for c in range(examples_per_class):
        ax = axes[r, c]
        ax.axis("off")
        if c >= len(cls_rows):
            continue
        row = cls_rows.iloc[c]
        p = Path(str(row["image_path"]))
        try:
            with Image.open(p) as im:
                ax.imshow(im)
            title = cls if c == 0 else str(row["id"])
            ax.set_title(title, fontsize=8)
        except Exception:
            ax.text(0.5, 0.5, "Unreadable", ha="center", va="center")

plt.tight_layout()
plt.show()